# Wang 5-Stack CNN training — binary

Trains the Wang et al. 5-Stack CNN baseline on the binary dataset for comparison with the proposed binary CNN.

Classes:
- `0`: no-rain
- `1`: rain, merging light, moderate, heavy and violent

Adaptations from the original paper:
- Output layer `Dense(1, sigmoid)` and `binary_crossentropy` loss;
- Input `(256, 256, 3)`: 256×256 RGB spectrograms;
- Same optimizer, hyperparameters and callbacks as the proposed binary CNN;
- Best model saved to `best_model_wang_binary.keras`.

In [ ]:
# Imports
import gc
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, LeakyReLU, BatchNormalization,
    MaxPooling2D, SpatialDropout2D,
    GlobalAveragePooling2D, Dense,
)
from tensorflow.keras.regularizers import l2
from data_pipeline import build_datasets, configure_gpu

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
# Parameters, same as the proposed binary CNN
CSV_PATH       = "Split70-15-15.csv"
EPOCHS         = 50
BATCH_SIZE     = 32
LEARNING_RATE  = 1e-3
SEED           = 42

PATIENCE_EARLY_STOPPING = 15
PATIENCE_REDUCE_LR      = 5
LR_REDUCE_FACTOR        = 0.5
LR_MIN                  = 1e-6


In [ ]:
# Garbage collection callback
class GarbageCollectionCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        collected = gc.collect()
        print(f"   [GC] {collected} objetos liberados.")


In [ ]:
# Wang et al. 5-Stack CNN with binary output: Dense(1, sigmoid) instead of Dense(5, softmax)

def build_wang_5stack_cnn_binary(input_shape=(256, 256, 3)):
    model = Sequential(name="Wang_5Stack_CNN_Binary")

    # L1 — Conv 7×7, stride 2, 64 filters
    model.add(Conv2D(64, (7, 7), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005), input_shape=input_shape))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())
    model.add(SpatialDropout2D(0.07))

    # L2 — Conv 5×5, stride 2, 48 filters + MaxPool
    model.add(Conv2D(48, (5, 5), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005)))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(SpatialDropout2D(0.07))

    # L3 — Conv 5×5, stride 2, 48 filters + MaxPool
    model.add(Conv2D(48, (5, 5), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005)))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(SpatialDropout2D(0.07))

    # L4 — Conv 3×3, stride 2, 32 filters
    model.add(Conv2D(32, (3, 3), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005)))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())
    model.add(SpatialDropout2D(0.14))

    # L5 — Conv 3×3, stride 2, 64 filters
    model.add(Conv2D(64, (3, 3), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005)))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())

    # Output — GAP + Dense(1, sigmoid)
    model.add(GlobalAveragePooling2D())
    model.add(Dense(1, activation='sigmoid'))

    return model


In [ ]:
# Data loading, model construction and compilation
configure_gpu()

train_ds, val_ds, test_ds = build_datasets(CSV_PATH, batch_size=BATCH_SIZE)

model = build_wang_5stack_cnn_binary(input_shape=(256, 256, 3))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

model.summary()


In [ ]:
# Callbacks, same as the proposed binary CNN
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_model_wang_binary.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE_EARLY_STOPPING,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=LR_REDUCE_FACTOR,
        patience=PATIENCE_REDUCE_LR,
        min_lr=LR_MIN,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(filename="training_history_wang_binary.csv"),
    GarbageCollectionCallback(),
]


In [ ]:
# Training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# Training summary
n_epochs_run  = len(history.history["loss"])
best_val_loss = min(history.history["val_loss"])
best_val_acc  = max(history.history["val_accuracy"])
best_val_auc  = max(history.history["val_auc"])

print(f"Épocas executadas:   {n_epochs_run} / {EPOCHS}")
print(f"Melhor val_loss:     {best_val_loss:.4f}")
print(f"Melhor val_accuracy: {best_val_acc:.4f}")
print(f"Melhor val_AUC:      {best_val_auc:.4f}")
print(f"Modelo salvo em:     best_model_wang_binary.keras")
print(f"Histórico em:        training_history_wang_binary.csv")


In [ ]:
# Learning curves

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Loss
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss — Wang 5-Stack CNN Binary")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss")
axes[0].legend()

# Accuracy
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim([0, 1]); axes[1].legend()

# AUC
axes[2].plot(history.history["auc"], label="train")
axes[2].plot(history.history["val_auc"], label="val")
axes[2].set_title("AUC-ROC")
axes[2].set_xlabel("Época"); axes[2].set_ylabel("AUC")
axes[2].set_ylim([0, 1]); axes[2].legend()

plt.tight_layout()
plt.show()

print("\n" + "=" * 40)
print("RESUMO FINAL")
print("=" * 40)
print(f"Épocas:        {n_epochs_run} / {EPOCHS}")
print(f"val_loss:      {best_val_loss:.4f}")
print(f"val_accuracy:  {best_val_acc:.4f}")
print(f"val_AUC:       {best_val_auc:.4f}")
print("=" * 40)
print()
print("PRÓXIMO PASSO: rode Wang_Binary_Evaluation.ipynb para métricas finais.")


In [ ]:
# Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss — Wang 5-Stack CNN Binary")
plt.xlabel("Época")
plt.legend()
plt.tight_layout()
plt.show()

# AUC
plt.figure(figsize=(8, 5))
plt.plot(history.history["auc"], label="train")
plt.plot(history.history["val_auc"], label="val")
plt.title("AUC — Wang 5-Stack CNN Binary")
plt.xlabel("Época")
plt.legend()
plt.tight_layout()
plt.show()

# Precision
plt.figure(figsize=(8, 5))
plt.plot(history.history["precision"], label="train")
plt.plot(history.history["val_precision"], label="val")
plt.title("Precision — Wang 5-Stack CNN Binary")
plt.xlabel("Época")
plt.ylim([0, 1])
plt.legend()
plt.tight_layout()
plt.show()

# Recall
plt.figure(figsize=(8, 5))
plt.plot(history.history["recall"], label="train")
plt.plot(history.history["val_recall"], label="val")
plt.title("Recall — Wang 5-Stack CNN Binary")
plt.xlabel("Época")
plt.ylim([0, 1])
plt.legend()
plt.tight_layout()
plt.show()

# Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy — Wang 5-Stack CNN Binary")
plt.xlabel("Época")
plt.ylim([0, 1])
plt.legend()
plt.tight_layout()
plt.show()

# Final summary
best_acc_idx = int(np.argmax(history.history["val_accuracy"]))
best_acc     = history.history["val_accuracy"][best_acc_idx]

print("=" * 40)
print("RESUMO FINAL")
print("=" * 40)
print(f"Épocas executadas: {n_epochs_run} / {EPOCHS}")
print(f"Melhor val_loss:   {best_val_loss:.4f}")
print(f"Melhor val_auc:    {best_val_auc:.4f}")
print(f"Melhor Accuracy:   {best_acc:.4f} (época {best_acc_idx + 1})")
print("=" * 40)